# Байесовский классификатор

Данная модель была придумана и спроектирована в рамках участия в Олимпиаде МГУ по финансовой культуре в 2026 году

(см. папку (прикреплена в резюме) с проектами в гугл диске)

Идея: на входы сети подаются статистические частоты, которые соответствуют частоте ответов респондентов по конкретному вопросу (в рамках задания по каждой из трех ключевых групп было выбрано несколько наиболее значимых вопросов из опросников), далее сеть рассчитывает вероятности по теореме Байеса на каждом узлу и выводит вероятности итогового индекса по трем его дискретным значениям (0, 0.5 и 1); далее находится математическое ожидание, которое и является значением искомого индекса

Преимущества подхода: можно смотреть на динамику компонентов с точки зрения их вероятностей

Итог работы показал рост данного индекса за последние несколько лет

In [ ]:
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [ ]:
# Определяем состояния
states = ['0', '0.5', '1'] # Дискретные состояния, которые может принять индекс
state_to_float = {'0': 0.0, '0.5': 0.5, '1': 1.0}

# Функция для создания CPD по правилу "шумное среднее"
def noisy_mean_cpd(variable, parents, model_states):
    """
    Создаёт TabularCPD для узла variable с заданными parents.
    Правило: значение ребёнка с вероятностью 0.8 равно ближайшему к среднему родителей,
    с вероятностью 0.1 каждое из двух остальных.
    """
    num_states = len(model_states)
    num_parent_combinations = np.prod([len(model_states)] * len(parents))
    cpd_array = np.zeros((num_states, num_parent_combinations))

    from itertools import product
    parent_combinations = list(product(model_states, repeat=len(parents)))

    for i, combo in enumerate(parent_combinations):
        mean_val = np.mean([state_to_float[s] for s in combo])
        if mean_val < 0.25:
            target = '0'
        elif mean_val < 0.75:
            target = '0.5'
        else:
            target = '1'
        for j, state in enumerate(model_states):
            if state == target:
                cpd_array[j, i] = 0.8
            else:
                cpd_array[j, i] = 0.1

    cpd = TabularCPD(variable, num_states, cpd_array,
                     evidence=parents, evidence_card=[len(model_states)]*len(parents))
    return cpd

# Создаём модель с изначально равномерными априорными распределениями
model = DiscreteBayesianNetwork()

# Узлы
nodes = ['x1', 'x2', 'x3', 'y1', 'y2', 'y3', 'b1', 'b2', 'c1', 'c2', 'c3', 'd1', 'd2',
         'B', 'C', 'D', 'A']
model.add_nodes_from(nodes)

# Рёбра
edges = [
    ('x1', 'b2'), ('x2', 'b2'), ('x3', 'b2'),
    ('y1', 'c2'), ('y2', 'c2'), ('y3', 'c2'),
    ('b1', 'B'), ('b2', 'B'),
    ('c1', 'C'), ('c2', 'C'), ('c3', 'C'),
    ('d1', 'D'), ('d2', 'D'),
    ('B', 'A'), ('C', 'A'), ('D', 'A')
]
model.add_edges_from(edges)

# Список корневых узлов (не имеют родителей в модели)
root_nodes = ['x1', 'x2', 'x3', 'y1', 'y2', 'y3', 'b1', 'c1', 'c3', 'd1', 'd2']

# Создаём равномерные CPD для корней
cpds = []
for node in root_nodes:
    cpd = TabularCPD(node, 3, [[1/3], [1/3], [1/3]])
    cpds.append(cpd)

# CPD для остальных узлов
cpds.append(noisy_mean_cpd('b2', ['x1', 'x2', 'x3'], states))
cpds.append(noisy_mean_cpd('c2', ['y1', 'y2', 'y3'], states))
cpds.append(noisy_mean_cpd('B', ['b1', 'b2'], states))
cpds.append(noisy_mean_cpd('C', ['c1', 'c2', 'c3'], states))
cpds.append(noisy_mean_cpd('D', ['d1', 'd2'], states))
cpds.append(noisy_mean_cpd('A', ['B', 'C', 'D'], states))

model.add_cpds(*cpds)

print("Базовая модель с равномерными априорными распределениями создана.")
print("Проверка модели:", model.check_model())


print("\n--- Ввод эмпирических вероятностей для корневых узлов ---")
print("Для каждого узла введите три числа (вероятности для значений 0, 0.5, 1), разделённых пробелом.")
print("Сумма вероятностей должна быть равна 1 (допускается небольшая погрешность).")

Базовая модель с равномерными априорными распределениями создана.
Проверка модели: True

--- Ввод эмпирических вероятностей для корневых узлов ---
Для каждого узла введите три числа (вероятности для значений 0, 0.5, 1), разделённых пробелом.
Сумма вероятностей должна быть равна 1 (допускается небольшая погрешность).


In [ ]:
new_cpds = []
for node in root_nodes:
    while True:
        user_input = input(f"\nВведите вероятности для узла {node} (например, 0.2 0.3 0.5): ").strip()
        parts = user_input.split()
        if len(parts) != 3:
            print("Ошибка: необходимо ввести ровно три числа. Повторите ввод.")
            continue
        try:
            probs = [float(p) for p in parts]
        except ValueError:
            print("Ошибка: введите числовые значения. Повторите ввод.")
            continue
        if abs(sum(probs) - 1.0) > 1e-6:
            print(f"Ошибка: сумма вероятностей равна {sum(probs):.4f}, должна быть 1. Повторите ввод.")
            continue
        # Проверка, что вероятности неотрицательные
        if any(p < 0 for p in probs):
            print("Ошибка: вероятности не могут быть отрицательными. Повторите ввод.")
            continue
        break

    # Создаём новый CPD для этого корневого узла
    # probs[0] -> P(0), probs[1] -> P(0.5), probs[2] -> P(1)
    cpd = TabularCPD(node, 3, [[probs[0]], [probs[1]], [probs[2]]])
    new_cpds.append(cpd)


Введите вероятности для узла x1 (например, 0.2 0.3 0.5): 0.0605 0.5981 0.3414

Введите вероятности для узла x2 (например, 0.2 0.3 0.5): 0.0028 0.0693 0.9279

Введите вероятности для узла x3 (например, 0.2 0.3 0.5): 0.0019 0.1145 0.8836

Введите вероятности для узла y1 (например, 0.2 0.3 0.5): 0.4056 0.4369 0.1575

Введите вероятности для узла y2 (например, 0.2 0.3 0.5): 0.4217 0.4296 0.1487

Введите вероятности для узла y3 (например, 0.2 0.3 0.5): 0.3105 0.5026 0.1869

Введите вероятности для узла b1 (например, 0.2 0.3 0.5): 0.3925 0.0 0.6075

Введите вероятности для узла c1 (например, 0.2 0.3 0.5): 0.1279 0.7371 0.135

Введите вероятности для узла c3 (например, 0.2 0.3 0.5): 0.1343 0.4162 0.4495

Введите вероятности для узла d1 (например, 0.2 0.3 0.5): 0.2079 0.0 0.7921

Введите вероятности для узла d2 (например, 0.2 0.3 0.5): 0.1275 0.8604 0.0121


In [ ]:
# Обновляем модель: добавляем все новые CPD (они заменят старые)
model.add_cpds(*new_cpds)

# Проверяем модель после обновления
print("\nМодель после обновления априорных распределений:")
print("Проверка модели:", model.check_model())

# Создаём объект для вывода
infer = VariableElimination(model)

# Пример запроса: маргинальное распределение A
print("\n--- Результаты расчёта ---")
print("Маргинальное распределение A (целевая переменная) с учётом введённых данных:")
result_A = infer.query(['A'])
print(result_A)


Модель после обновления априорных распределений:
Проверка модели: True

--- Результаты расчёта ---
Маргинальное распределение A (целевая переменная) с учётом введённых данных:
+------+----------+
| A    |   phi(A) |
+======+==========+
| A(0) |   0.1171 |
+------+----------+
| A(1) |   0.5702 |
+------+----------+
| A(2) |   0.3127 |
+------+----------+


In [ ]:
# Дополнительно можно спросить пользователя, хочет ли он получить вероятности других узлов
while True:
    ans = input("\nХотите рассчитать вероятности для другого узла? (да/нет): ").strip().lower()
    if ans not in ['да', 'yes', 'y', 'нет', 'no', 'n']:
        print("Пожалуйста, ответьте 'да' или 'нет'.")
        continue
    if ans in ['нет', 'no', 'n']:
        break
    target = input("Введите имя узла (например, B, C, b2, x1 и т.д.): ").strip()
    if target not in nodes:
        print("Такого узла нет в модели. Попробуйте ещё раз.")
        continue
    result = infer.query([target])
    print(f"\nРаспределение {target}:")
    print(result)


Хотите рассчитать вероятности для другого узла? (да/нет): да
Введите имя узла (например, B, C, b2, x1 и т.д.): B

Распределение B:
+------+----------+
| B    |   phi(B) |
+======+==========+
| B(0) |   0.1106 |
+------+----------+
| B(1) |   0.3729 |
+------+----------+
| B(2) |   0.5166 |
+------+----------+

Хотите рассчитать вероятности для другого узла? (да/нет): lf
Пожалуйста, ответьте 'да' или 'нет'.

Хотите рассчитать вероятности для другого узла? (да/нет): да
Введите имя узла (например, B, C, b2, x1 и т.д.): C

Распределение C:
+------+----------+
| C    |   phi(C) |
+======+==========+
| C(0) |   0.1562 |
+------+----------+
| C(1) |   0.6852 |
+------+----------+
| C(2) |   0.1586 |
+------+----------+

Хотите рассчитать вероятности для другого узла? (да/нет): x1
Пожалуйста, ответьте 'да' или 'нет'.

Хотите рассчитать вероятности для другого узла? (да/нет): да
Введите имя узла (например, B, C, b2, x1 и т.д.): x1

Распределение x1:
+-------+-----------+
| x1    |   phi(x1) |
